# Compare datasets

In [1]:
import anndata as ad

# Annes data
#h5ad_file='../data/blood/10x-rep1-kallisto-cellbender/10x-rep1-kallisto-cellbender'
#h5ad_file='../blood/10x-rep2-kallisto-cellbender/10x-rep2-kallisto-cellbender'
#h5ad_file='../blood/bd-rhap-rep1/bd-rhap-rep1'
#h5ad_file='../blood/bd-rhap-rep2/bd-rhap-rep2'

# Human immune health atlas
h5ad_file='../data/human_immune_health_atlas/human_immune_health_atlas_full'

adata_backed = ad.io.read_h5ad(f'{h5ad_file}.h5ad', backed="r")
adata_a = adata_backed[:3000].raw.to_adata()
umi_df_a = adata_a.to_df()#layer="counts")
adata_backed.file.close()

# Celltypist data
adata_b = ad.io.read_h5ad('../data/CellTypistDataset/global.h5ad')
adata_b = adata_b[adata_b.obs['Organ'] == 'BLD'].copy()
adata_b.layers["counts"] = adata_b.X.copy()
umi_df_b = adata_b.to_df()

In [2]:
print(adata_a)
print("---")
print(adata_b)

AnnData object with n_obs × n_vars = 3000 × 33538
    obs: 'cohort.cohortGuid', 'sample.sampleKitGuid', 'specimen.specimenGuid', 'pipeline.fileGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.ageGroup', 'subject.race', 'subject.ethnicity', 'subject.cmv', 'subject.bmi', 'sample.visitName', 'sample.drawYear', 'sample.subjectAgeAtDraw', 'batch_id', 'pool_id', 'chip_id', 'well_id', 'barcodes', 'original_barcodes', 'cell_name', 'n_reads', 'n_umis', 'n_genes', 'total_counts_mito', 'pct_counts_mito', 'doublet_score', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3'
    var: 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'AIFI_L1_colors', 'AIFI_L2_colors', 'AIFI_L3_colors', 'celltypist.low_colors', 'hvg', 'keep_colors', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'seurat.l2.5_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    obsp: 'con

In [3]:
import pandas as pd

def compare_dataframe_features(df1, df2):
    """
    Vergleicht die Spalten (Features) von zwei Pandas DataFrames.
    Gibt die Anzahl der Gemeinsamkeiten und Unterschiede aus und 
    gibt die entsprechenden Sets zurück.
    """
    # Spaltennamen als Sets extrahieren
    features1 = set(df1.columns)
    features2 = set(df2.columns)
    
    # Gemeinsamkeiten und Unterschiede berechnen
    common_features = features1.intersection(features2)
    unique_to_df1 = features1 - features2
    unique_to_df2 = features2 - features1
    all_differences = features1.symmetric_difference(features2)
    
    # Ergebnisse ausgeben
    print("=== Feature-Vergleich ===")
    print(f"Gemeinsame Features: {len(common_features)}")
    print(f"Unterschiedliche Features (insgesamt): {len(all_differences)}")
    print(f"  ↳ Nur in DF1 vorhanden: {len(unique_to_df1)}")
    print(f"  ↳ Nur in DF2 vorhanden: {len(unique_to_df2)}")
    
    # Die tatsächlichen Namen für die weitere Verarbeitung zurückgeben
    return {
        "common": common_features,
        "unique_df1": unique_to_df1,
        "unique_df2": unique_to_df2,
        "all_differences": all_differences
    }




# Compare Dataframes
ergebnisse = compare_dataframe_features(umi_df_a, umi_df_b)

# Du kannst dir die genauen Namen auch anzeigen lassen:
print("\nDie gemeinsamen Features sind:", ergebnisse["common"])

=== Feature-Vergleich ===
Gemeinsame Features: 31915
Unterschiedliche Features (insgesamt): 6309
  ↳ Nur in DF1 vorhanden: 1623
  ↳ Nur in DF2 vorhanden: 4686

Die gemeinsamen Features sind: {'PIAS4', 'AP001469.1', 'IFIT5', 'LINC01150', 'AC095059.1', 'LAMP1', 'STRA8', 'FGFR1OP', 'HPGD', 'EHD2', 'AC006482.1', 'TMEM156', 'FAHD2B', 'AC008567.2', 'FUCA1', 'ZFP37', 'ZNF738', 'ARL8A', 'SAMD4A', 'EPHB4', 'UGT2B15', 'AP001020.1', 'TVP23A', 'PRR30', 'PPBP', 'AL590677.1', 'FTCDNL1', 'TET1', 'SPIRE2', 'AC087672.2', 'AC024267.6', 'HJV', 'AC022028.2', 'BHMG1', 'IGLV5-48', 'ARL6IP5', 'RHOBTB1', 'RAB3D', 'AC004946.1', 'ARMCX4', 'PGM5P3-AS1', 'SCGB1D1', 'PRR35', 'IGKV1OR22-5', 'CLDN23', 'AC145141.2', 'MYBL1', 'AC006111.1', 'SRRD', 'AC007193.2', 'RPP25L', 'CAMK4', 'AL161729.1', 'MEA1', 'MAPRE1', 'LINC01358', 'IGKV1D-27', 'TTYH1', 'AC008555.1', 'PDE2A', 'AC006159.2', 'TEAD4', 'AC012506.2', 'PHKB', 'AC116609.3', 'SHBG', 'EIF1AX', 'TRAV1-2', 'LINC02154', 'CXCL16', 'AC120498.9', 'AC100793.4', 'SNAPC5', 'AC

In [4]:
import json
import pandas as pd

def check_marker_genes_from_json(df, json_path):
    """
    Lädt Marker-Gene aus einer JSON-Datei und prüft, welche davon
    in den Spalten (Features) des DataFrames vorhanden sind.
    
    Parameters:
    df (pd.DataFrame): Der zu prüfende DataFrame.
    json_path (str): Pfad zur JSON-Datei mit den Marker-Genen.
    
    Returns:
    dict: Ein Wörterbuch mit detaillierten Ergebnissen pro Kategorie/Zelltyp.
    """
    # 1. JSON-Datei mit den Marker-Genen laden
    try:
        with open(json_path, 'r', encoding='utf-8') as file:
            marker_dict = json.load(file)
    except FileNotFoundError:
        print(f"Fehler: Die Datei '{json_path}' wurde nicht gefunden.")
        return None
    except json.JSONDecodeError:
        print(f"Fehler: Die Datei '{json_path}' ist kein gültiges JSON.")
        return None

    # Spaltennamen des DataFrames als Set für ultraschnelle Abgleiche
    df_features = set(df.columns)
    
    ergebnisse = {}
    
    print("=== Marker-Gen Analyse ===")
    
    # 2. Iteration über die Kategorien (z.B. Zelltypen) im JSON
    for kategorie, gene_liste in marker_dict.items():
        # + und - am Ende der Gene entfernen
        gene_set: set[str] = set()
        for g in gene_liste:
            if g.endswith("+") or g.endswith("-"):
                gene_set.add(g[:-1])
            else:
                gene_set.add(g)
        
        # Schnittmenge: Welche Gene sind im DataFrame?
        gefundene_gene = gene_set.intersection(df_features)
        # Differenz: Welche Gene fehlen im DataFrame?
        fehlende_gene = gene_set - df_features
        
        # Prozentualer Anteil der gefundenen Gene berechnen
        gesamt_anzahl = len(gene_set)
        abdeckungsgrad = (len(gefundene_gene) / gesamt_anzahl * 100) if gesamt_anzahl > 0 else 0
        
        # Ergebnisse für diese Kategorie speichern
        ergebnisse[kategorie] = {
            "gefunden": gefundene_gene,
            "fehlend": fehlende_gene,
            "abdeckung_prozent": round(abdeckungsgrad, 2)
        }
        
        # Übersichtliche Ausgabe in der Konsole
        print(f"\nKategorie/Zelltyp: {kategorie}")
        print(f"  -> Abdeckung: {len(gefundene_gene)}/{gesamt_anzahl} ({abdeckungsgrad:.1f}%)")
        print(f"  -> Gefunden: {list(gefundene_gene) if gefundene_gene else 'Keine'}")
        if fehlende_gene:
            print(f"  -> Fehlend: {list(fehlende_gene)}")
            
    return ergebnisse


analyse_resultat = check_marker_genes_from_json(umi_df_a, '../scumi-dev/R/marker_gene/human_pbmc_marker.json')

=== Marker-Gen Analyse ===

Kategorie/Zelltyp: CD4+ T cell
  -> Abdeckung: 6/13 (46.2%)
  -> Gefunden: ['CST7', 'NKG7', 'GNLY', 'CD8A', 'CD8B', 'IL7R']
  -> Fehlend: ['CD4', 'CD27', 'CD3E', 'TRAC', 'CD3G', 'TCF7', 'CD3D']

Kategorie/Zelltyp: Cytotoxic T cell
  -> Abdeckung: 6/11 (54.5%)
  -> Gefunden: ['FCER1G', 'CCL5', 'CD8A', 'CD8B', 'GZMK', 'NKG7']
  -> Fehlend: ['CD4', 'CD3E', 'TRAC', 'CD3G', 'CD3D']

Kategorie/Zelltyp: B cell
  -> Abdeckung: 7/7 (100.0%)
  -> Gefunden: ['CD19', 'CD79A', 'CD79B', 'MS4A1', 'IGHM', 'MZB1', 'IGHD']

Kategorie/Zelltyp: Natural killer cell
  -> Abdeckung: 13/19 (68.4%)
  -> Gefunden: ['FCGR3A', 'FCER1G', 'KLRC3', 'NCAM1', 'ITGAM', 'KLRC4', 'KLRF1', 'CD14', 'KLRD1', 'KLRC2', 'KLRB1', 'NKG7', 'KLRC1']
  -> Fehlend: ['ITGAL', 'CD3E', 'TRAC', 'CD3G', 'FCGR3B', 'CD3D']

Kategorie/Zelltyp: CD14+ monocyte
  -> Abdeckung: 13/22 (59.1%)
  -> Gefunden: ['CSF3R', 'FCGR3A', 'CX3CR1', 'CSF1R', 'VCAN', 'FCN1', 'S100A12', 'ITGAM', 'TYROBP', 'CD14', 'KLRD1', 'KLRB1', '

In [4]:
# raw auf human immune atlas
import json
import pandas as pd

def check_marker_genes_from_json(df, json_path):
    """
    Lädt Marker-Gene aus einer JSON-Datei und prüft, welche davon
    in den Spalten (Features) des DataFrames vorhanden sind.
    
    Parameters:
    df (pd.DataFrame): Der zu prüfende DataFrame.
    json_path (str): Pfad zur JSON-Datei mit den Marker-Genen.
    
    Returns:
    dict: Ein Wörterbuch mit detaillierten Ergebnissen pro Kategorie/Zelltyp.
    """
    # 1. JSON-Datei mit den Marker-Genen laden
    try:
        with open(json_path, 'r', encoding='utf-8') as file:
            marker_dict = json.load(file)
    except FileNotFoundError:
        print(f"Fehler: Die Datei '{json_path}' wurde nicht gefunden.")
        return None
    except json.JSONDecodeError:
        print(f"Fehler: Die Datei '{json_path}' ist kein gültiges JSON.")
        return None

    # Spaltennamen des DataFrames als Set für ultraschnelle Abgleiche
    df_features = set(df.columns)
    
    ergebnisse = {}
    
    print("=== Marker-Gen Analyse ===")
    
    # 2. Iteration über die Kategorien (z.B. Zelltypen) im JSON
    for kategorie, gene_liste in marker_dict.items():
        # + und - am Ende der Gene entfernen
        gene_set: set[str] = set()
        for g in gene_liste:
            if g.endswith("+") or g.endswith("-"):
                gene_set.add(g[:-1])
            else:
                gene_set.add(g)
        
        # Schnittmenge: Welche Gene sind im DataFrame?
        gefundene_gene = gene_set.intersection(df_features)
        # Differenz: Welche Gene fehlen im DataFrame?
        fehlende_gene = gene_set - df_features
        
        # Prozentualer Anteil der gefundenen Gene berechnen
        gesamt_anzahl = len(gene_set)
        abdeckungsgrad = (len(gefundene_gene) / gesamt_anzahl * 100) if gesamt_anzahl > 0 else 0
        
        # Ergebnisse für diese Kategorie speichern
        ergebnisse[kategorie] = {
            "gefunden": gefundene_gene,
            "fehlend": fehlende_gene,
            "abdeckung_prozent": round(abdeckungsgrad, 2)
        }
        
        # Übersichtliche Ausgabe in der Konsole
        print(f"\nKategorie/Zelltyp: {kategorie}")
        print(f"  -> Abdeckung: {len(gefundene_gene)}/{gesamt_anzahl} ({abdeckungsgrad:.1f}%)")
        print(f"  -> Gefunden: {list(gefundene_gene) if gefundene_gene else 'Keine'}")
        if fehlende_gene:
            print(f"  -> Fehlend: {list(fehlende_gene)}")
            
    return ergebnisse


analyse_resultat = check_marker_genes_from_json(umi_df_a, '../scumi-dev/R/marker_gene/human_pbmc_marker.json')

=== Marker-Gen Analyse ===

Kategorie/Zelltyp: CD4+ T cell
  -> Abdeckung: 13/13 (100.0%)
  -> Gefunden: ['CD27', 'CD4', 'CD3E', 'CD8B', 'TCF7', 'CD3D', 'CD8A', 'CD3G', 'CST7', 'IL7R', 'GNLY', 'NKG7', 'TRAC']

Kategorie/Zelltyp: Cytotoxic T cell
  -> Abdeckung: 11/11 (100.0%)
  -> Gefunden: ['FCER1G', 'CD4', 'CD3E', 'CD8B', 'CD8A', 'CD3D', 'CD3G', 'GZMK', 'CCL5', 'NKG7', 'TRAC']

Kategorie/Zelltyp: B cell
  -> Abdeckung: 7/7 (100.0%)
  -> Gefunden: ['MS4A1', 'CD19', 'MZB1', 'IGHM', 'CD79B', 'CD79A', 'IGHD']

Kategorie/Zelltyp: Natural killer cell
  -> Abdeckung: 19/19 (100.0%)
  -> Gefunden: ['NCAM1', 'KLRF1', 'FCGR3B', 'KLRC2', 'ITGAM', 'FCER1G', 'KLRD1', 'CD3D', 'FCGR3A', 'NKG7', 'KLRC1', 'CD3G', 'ITGAL', 'KLRB1', 'KLRC3', 'CD14', 'TRAC', 'KLRC4', 'CD3E']

Kategorie/Zelltyp: CD14+ monocyte
  -> Abdeckung: 22/22 (100.0%)
  -> Gefunden: ['CSF3R', 'FCGR3B', 'FCN1', 'ITGAM', 'TYROBP', 'S100A12', 'KLRD1', 'CD3D', 'FCGR3A', 'S100A8', 'NKG7', 'CD3G', 'VCAN', 'ITGAL', 'S100A9', 'KLRB1', 'CD1

In [5]:
analyse_resultat = check_marker_genes_from_json(umi_df_b, '../scumi-dev/R/marker_gene/human_pbmc_marker.json')

=== Marker-Gen Analyse ===

Kategorie/Zelltyp: CD4+ T cell
  -> Abdeckung: 13/13 (100.0%)
  -> Gefunden: ['CST7', 'TCF7', 'CD4', 'CD27', 'CD3E', 'CD3D', 'TRAC', 'IL7R', 'CD3G', 'GNLY', 'CD8A', 'CD8B', 'NKG7']

Kategorie/Zelltyp: Cytotoxic T cell
  -> Abdeckung: 11/11 (100.0%)
  -> Gefunden: ['CD4', 'FCER1G', 'CD3E', 'CD3D', 'TRAC', 'CCL5', 'CD3G', 'CD8A', 'CD8B', 'GZMK', 'NKG7']

Kategorie/Zelltyp: B cell
  -> Abdeckung: 7/7 (100.0%)
  -> Gefunden: ['CD19', 'CD79A', 'CD79B', 'MS4A1', 'IGHM', 'MZB1', 'IGHD']

Kategorie/Zelltyp: Natural killer cell
  -> Abdeckung: 19/19 (100.0%)
  -> Gefunden: ['CD3E', 'KLRC4', 'FCGR3B', 'TRAC', 'KLRC2', 'KLRB1', 'ITGAL', 'KLRC3', 'NCAM1', 'KLRF1', 'CD14', 'NKG7', 'CD3D', 'FCER1G', 'ITGAM', 'CD3G', 'KLRD1', 'KLRC1', 'FCGR3A']

Kategorie/Zelltyp: CD14+ monocyte
  -> Abdeckung: 22/22 (100.0%)
  -> Gefunden: ['CD3E', 'FCN1', 'S100A12', 'TYROBP', 'FCGR3B', 'S100A8', 'TRAC', 'KLRB1', 'ITGAL', 'CD14', 'NKG7', 'CD3D', 'CSF3R', 'CX3CR1', 'CSF1R', 'VCAN', 'LYZ', 